# ARES Terrain AI — batch image understanding

Run in Google Colab to analyze one or more JPG, JPEG, PNG, or WEBP images with pretrained SegFormer B0 and Depth Anything V2 Small. Each image is processed sequentially, resized with aspect ratio preserved to a maximum side of 1024 pixels, and exported with semantic and relative-depth data for future mapping work.

Depth is relative and uncalibrated: it is not measured physical distance or true surface elevation. Semantic labels are model predictions and do not establish landing safety; a predicted road or floor may still be unsuitable. No training or fine-tuning is performed. A single upload continues to work as a one-image batch.


In [ ]:
# Confirm Google Colab and select an available accelerator. CPU fallback is supported.
try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError('Open this notebook in Google Colab to use the upload workflow.') from exc

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Inference device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU detected; inference will use CPU and may take longer.')


In [ ]:
# Colab provides PyTorch, NumPy, Pillow, and Matplotlib. Install only Transformers.
%pip install -q transformers==4.49.0


In [ ]:
from pathlib import Path
import colorsys
import gc
import io
import json
import shutil
import tempfile
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from PIL import Image, ImageOps
from IPython.display import display
from google.colab import files
from transformers import (
    AutoImageProcessor,
    AutoModelForDepthEstimation,
    AutoModelForSemanticSegmentation,
)

SEGMENTATION_MODEL_ID = 'nvidia/segformer-b0-finetuned-ade-512-512'
DEPTH_MODEL_ID = 'depth-anything/Depth-Anything-V2-Small-hf'
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
MAX_IMAGE_DIMENSION = 1024
OVERLAY_ALPHA = 0.45


def build_terrain_metadata(filename, source_dimensions, image_dimensions,
                           segmentation_model_id, depth_model_id,
                           detected_classes, segmentation_palette,
                           depth_min, depth_max):
    return {
        'schema_version': 'ares.terrain.v1',
        'filename': str(filename),
        'image_dimensions': {'width': int(image_dimensions[0]), 'height': int(image_dimensions[1])},
        'source_dimensions': {'width': int(source_dimensions[0]), 'height': int(source_dimensions[1])},
        'segmentation': {
            'model_id': segmentation_model_id,
            'classes': detected_classes,
            'palette_rgb_hex_by_class_id': segmentation_palette,
        },
        'depth': {
            'model_id': depth_model_id,
            'kind': 'relative_depth',
            'calibration_status': 'uncalibrated_relative_depth',
            'physical_distance_available': False,
            'raw_dtype': 'float32',
            'range': {'minimum': float(depth_min), 'maximum': float(depth_max)},
            'visualization_normalization': {
                'method': 'per_image_min_max', 'minimum': 0, 'maximum': 255,
                'source_minimum': float(depth_min), 'source_maximum': float(depth_max),
            },
        },
        'coordinate_and_measurement_assumptions': {
            'coordinate_frame': '2d_image_pixels',
            'origin': 'top_left',
            'x_direction': 'right',
            'y_direction': 'down',
            'dimensions_refer_to': 'resized exported image; aspect ratio preserved',
            'georeferencing_available': False,
            'camera_intrinsics_available': False,
            'physical_scale_available': False,
            'depth_is_surface_elevation': False,
            'assumptions': [
                'Pixel coordinates refer to the resized exported image.',
                'Relative depth has no calibrated distance scale.',
                'No geospatial frame, camera calibration, or physical surface measurement is provided.',
            ],
        },
    }


def fit_image_dimensions(width, height, maximum):
    if width <= 0 or height <= 0 or maximum <= 0:
        raise ValueError('Image dimensions and maximum must be positive.')
    scale = min(1.0, maximum / max(width, height))
    return max(1, round(width * scale)), max(1, round(height * scale))

def pop_next_upload(upload_queue):
    filename = next(iter(upload_queue))
    return filename, upload_queue.pop(filename)


def predicted_class_ids(segmentation_logits):
    return segmentation_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)


def raw_depth_float32(depth_values):
    return depth_values.float().cpu().numpy().astype(np.float32, copy=False)


def create_output_archive(output_root, archive_path):
    output_root = Path(output_root)
    archive_path = Path(archive_path)
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for output_path in sorted(output_root.rglob('*')):
            if output_path.is_file():
                archive_name = Path('ares_terrain_outputs') / output_path.relative_to(output_root)
                archive.write(output_path, arcname=archive_name.as_posix())
    return archive_path

print('Imports and export helpers ready.')


In [ ]:
# Upload one or more images. A one-image upload remains a valid workflow.
try:
    uploaded_files = files.upload()
    if not uploaded_files:
        raise ValueError('Upload at least one JPG, JPEG, PNG, or WEBP image.')
    unsupported = [name for name in uploaded_files if Path(name).suffix.lower() not in ALLOWED_EXTENSIONS]
    if unsupported:
        print('These unsupported files will be skipped:', ', '.join(unsupported))
    OUTPUT_ROOT = Path(tempfile.mkdtemp(prefix='ares_terrain_outputs_'))
    ZIP_PATH = Path('/content/ares_terrain_outputs.zip')
    print(f'Received {len(uploaded_files)} file(s). Processing supported images sequentially.')
except Exception as exc:
    raise RuntimeError(f'Image upload failed: {exc}') from exc


In [ ]:
# Load SegFormer once. Model weights remain in the Colab runtime, not the repository.
try:
    if 'segmentation_model' not in globals():
        segmentation_processor = AutoImageProcessor.from_pretrained(SEGMENTATION_MODEL_ID)
        segmentation_model = AutoModelForSemanticSegmentation.from_pretrained(SEGMENTATION_MODEL_ID).to(DEVICE).eval()
except Exception as exc:
    raise RuntimeError(f'Could not load SegFormer model {SEGMENTATION_MODEL_ID}: {exc}') from exc
print('SegFormer B0 is ready.')


In [ ]:
# Load Depth Anything once. Its predictions are relative, not calibrated distances.
try:
    if 'depth_model' not in globals():
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_ID)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_ID).to(DEVICE).eval()
except Exception as exc:
    raise RuntimeError(f'Could not load depth model {DEPTH_MODEL_ID}: {exc}') from exc
print('Depth Anything V2 Small is ready.')


In [ ]:
# Process images one at a time to limit intermediate GPU and host memory use.
# The queue is consumed as we go, releasing each uploaded byte buffer after processing.
summary = {
    'schema_version': 'ares.terrain.batch.v1',
    'segmentation_model_id': SEGMENTATION_MODEL_ID,
    'depth_model_id': DEPTH_MODEL_ID,
    'depth_calibration_status': 'uncalibrated_relative_depth',
    'processed_images': [],
    'failed_images': [],
}
image_count = len(uploaded_files)
for image_number in range(1, image_count + 1):
    filename, image_bytes = pop_next_upload(uploaded_files)
    image_directory_name = f'image_{image_number:03d}'
    image_directory = OUTPUT_ROOT / image_directory_name
    seg_inputs = seg_logits = depth_inputs = depth_output = None
    try:
        if Path(filename).suffix.lower() not in ALLOWED_EXTENSIONS:
            raise ValueError('Unsupported image type; expected JPG, JPEG, PNG, or WEBP.')
        with Image.open(io.BytesIO(image_bytes)) as opened_image:
            oriented_image = ImageOps.exif_transpose(opened_image)
            source_dimensions = oriented_image.size
            target_size = fit_image_dimensions(source_dimensions[0], source_dimensions[1], MAX_IMAGE_DIMENSION)
            if target_size != source_dimensions:
                oriented_image = oriented_image.resize(target_size, resample=Image.Resampling.LANCZOS)
            original_image = oriented_image.convert('RGB')
        image_width, image_height = original_image.size
        image_directory.mkdir(parents=True, exist_ok=True)
        original_image.save(image_directory / 'original_image.png')
        print(f'[{image_number}/{image_count}] {filename}: {image_width} × {image_height} px')

        # Segmentation: argmax preserves the model's predicted ADE class IDs.
        seg_inputs = segmentation_processor(images=original_image, return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            seg_logits = segmentation_model(**seg_inputs).logits
            native_class_ids = predicted_class_ids(seg_logits)
        del seg_inputs, seg_logits
        seg_inputs = seg_logits = None
        class_id_image = Image.fromarray(native_class_ids, mode='L')
        class_id_image = class_id_image.resize(original_image.size, resample=Image.Resampling.NEAREST)
        class_id_mask = np.asarray(class_id_image, dtype=np.uint8)
        class_id_image.save(image_directory / 'segmentation_class_ids.png')

        # Resolve labels from SegFormer config; predictions and IDs are never fabricated.
        id_to_label = {int(class_id): str(label) for class_id, label in segmentation_model.config.id2label.items()}
        terrain_colors = {
            'road': '#808080', 'tree': '#228B22', 'grass': '#90EE90',
            'building': '#F28E2B', 'sky': '#4A90E2', 'earth': '#8B5A2B',
        }
        def color_for_class(class_id):
            label = id_to_label.get(int(class_id), f'class {class_id}')
            if label.strip().lower() in terrain_colors:
                return terrain_colors[label.strip().lower()]
            hue = (int(class_id) * 0.618033988749895) % 1.0
            rgb = colorsys.hsv_to_rgb(hue, 0.72, 0.95)
            return '#{:02X}{:02X}{:02X}'.format(*(round(value * 255) for value in rgb))
        detected_ids, pixel_counts = np.unique(class_id_mask, return_counts=True)
        detected = sorted(zip(detected_ids, pixel_counts), key=lambda pair: pair[1], reverse=True)
        class_records = []
        palette_by_class_id = {}
        color_lut = np.zeros((max(max(id_to_label, default=0), int(class_id_mask.max())) + 1, 3), dtype=np.uint8)
        for class_id in range(len(color_lut)):
            hex_color = color_for_class(class_id)
            color_lut[class_id] = (np.array(plt.matplotlib.colors.to_rgb(hex_color)) * 255).round().astype(np.uint8)
        for class_id, pixel_count in detected:
            class_id = int(class_id)
            label = id_to_label.get(class_id, f'class {class_id}')
            percentage = 100.0 * int(pixel_count) / int(class_id_mask.size)
            palette_by_class_id[str(class_id)] = color_for_class(class_id)
            class_records.append({'class_id': class_id, 'label': label, 'pixel_percentage': percentage})
        segmentation_rgb = color_lut[class_id_mask]
        segmentation_map = Image.fromarray(segmentation_rgb, mode='RGB')
        overlay = Image.blend(original_image, segmentation_map, alpha=OVERLAY_ALPHA)
        segmentation_map.save(image_directory / 'segmentation_mask.png')
        overlay.save(image_directory / 'segmentation_overlay.png')

        # Save the full-resolution float32 depth array and a separate normalized PNG.
        depth_inputs = depth_processor(images=original_image, return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            depth_output = depth_model(**depth_inputs).predicted_depth
            resized_depth = torch.nn.functional.interpolate(
                depth_output.unsqueeze(1), size=(image_height, image_width),
                mode='bicubic', align_corners=False
            ).squeeze(1)[0]
            raw_depth = raw_depth_float32(resized_depth)
        del depth_inputs, depth_output, resized_depth
        depth_inputs = depth_output = None
        if not np.isfinite(raw_depth).all():
            raise ValueError('Depth prediction contained non-finite values and cannot be exported as valid metadata.')
        depth_min, depth_max = float(raw_depth.min()), float(raw_depth.max())
        depth_span = depth_max - depth_min
        normalized_depth = (raw_depth - depth_min) / depth_span if depth_span > 1e-8 else np.zeros_like(raw_depth)
        depth_visualization = Image.fromarray((normalized_depth * 255).round().astype(np.uint8), mode='L')
        depth_visualization.save(image_directory / 'depth_map.png')
        np.save(image_directory / 'depth_raw.npy', raw_depth.astype(np.float32, copy=False))

        metadata = build_terrain_metadata(
            filename=filename,
            source_dimensions=source_dimensions,
            image_dimensions=original_image.size,
            segmentation_model_id=SEGMENTATION_MODEL_ID,
            depth_model_id=DEPTH_MODEL_ID,
            detected_classes=class_records,
            segmentation_palette=palette_by_class_id,
            depth_min=depth_min,
            depth_max=depth_max,
        )
        with (image_directory / 'terrain_metadata.json').open('w', encoding='utf-8') as metadata_file:
            json.dump(metadata, metadata_file, indent=2, allow_nan=False)
        # Display this image's results before processing the next upload.
        fig = plt.figure(figsize=(16, 8))
        grid = fig.add_gridspec(2, 3, height_ratios=[1, 0.85])
        axes = [fig.add_subplot(grid[0, column]) for column in range(3)]
        axes[0].imshow(original_image); axes[0].set_title(f'Original · {filename}')
        axes[1].imshow(segmentation_map); axes[1].set_title('Colour-coded segmentation')
        axes[2].imshow(overlay); axes[2].set_title('Segmentation overlay')
        depth_axis = fig.add_subplot(grid[1, 0])
        depth_axis.imshow(depth_visualization, cmap='inferno'); depth_axis.set_title('Relative depth (not metric)')
        legend_axis = fig.add_subplot(grid[1, 1:]); legend_axis.axis('off')
        legend_axis.set_title('Detected classes — pixel share', loc='left')
        legend_items = [Patch(facecolor=palette_by_class_id[str(record['class_id'])], edgecolor='none',
                              label=f"{record['label']} · {record['pixel_percentage']:.2f}%")
                        for record in class_records]
        if legend_items:
            legend_axis.legend(handles=legend_items, loc='upper left', ncol=2, frameon=False, fontsize=8)
        for axis in axes + [depth_axis]: axis.axis('off')
        fig.tight_layout()
        display(fig)
        plt.close(fig)
        print(f"Saved {image_directory_name}; depth is uncalibrated relative depth ({depth_min:.4f} to {depth_max:.4f}).")
        summary['processed_images'].append({
            'filename': filename,
            'directory': image_directory_name,
            'image_dimensions': metadata['image_dimensions'],
            'detected_class_count': len(class_records),
            'depth_range': metadata['depth']['range'],
        })
    except Exception as exc:
        if image_directory.exists():
            shutil.rmtree(image_directory)
        summary['failed_images'].append({'filename': filename, 'directory': image_directory_name, 'error': str(exc)})
        print(f'Failed {filename}: {exc}')
    finally:
        plt.close('all')
        for variable_name in ('image_bytes', 'oriented_image', 'original_image', 'class_id_image',
                              'class_id_mask', 'native_class_ids', 'segmentation_rgb', 'segmentation_map', 'overlay',
                              'raw_depth', 'normalized_depth', 'depth_visualization', 'metadata',
                              'id_to_label', 'terrain_colors', 'detected_ids', 'pixel_counts',
                              'detected', 'class_records', 'palette_by_class_id', 'color_lut',
                              'source_dimensions', 'target_size', 'image_width', 'image_height', 'image_directory',
                              'fig', 'grid', 'axes', 'depth_axis', 'legend_axis', 'legend_items',
                              'seg_inputs', 'seg_logits', 'depth_inputs', 'depth_output', 'resized_depth'):
            globals().pop(variable_name, None)
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
if not summary['processed_images']:
    raise RuntimeError('No images were processed successfully; no output archive was created.')
create_output_archive(OUTPUT_ROOT, ZIP_PATH)
print(f"Batch complete: {len(summary['processed_images'])} succeeded, {len(summary['failed_images'])} failed.")
print('Output archive:', ZIP_PATH)
files.download(str(ZIP_PATH))
